In [ ]:
# ruff: noqa: E402
import os
import subprocess
import sys
from pathlib import Path

from dotenv import load_dotenv

ENV_PATH = None
assert ENV_PATH is not None, "Environment file path is not set. Please set the ENV_PATH variable."
load_dotenv(dotenv_path=ENV_PATH)

import gbd_foodservice_insights_lab as foodservice_insights_lab

package_module_dir = Path(foodservice_insights_lab.__file__).resolve().parent

from gbd_foodservice_insights.plotting_utils import GBD_colors
from gbd_foodservice_insights_lab.extraction.pdf import (
    check_duplicates,
    check_extraction_by_page,
    check_high_duplicate_pages,
    check_missing_literal_values,
    combine_extracted_pdf_pages,
    find_most_deviating_file_page_combos,
    find_possible_misspellings,
    identify_unique_product_names,
    plot_bar_by_pdf,
    plot_rowcount_against_pagenum,
    plot_rows_by_page_for_each_pdf,
    plot_total_rows_by_page_across_pdfs,
    run_pdf_extraction_pipeline_multiple_files,
    run_pdf_preflight_sample,
    sample_files_and_rows,
    summarize_pdf_validation_data,
)
from gbd_foodservice_insights_lab.notebook_runscript_setup import (
    detect_client_structure,
    save_client_metadata,
    setup_api_clients,
    setup_pandas_display,
)

# Configure pandas display
setup_pandas_display()

# Initialize API clients
clients = setup_api_clients(openai=True, whisper=True, gemini=True)
OpenAI_client = clients["openai_client"]
whisper_client = clients["whisper_client"]
gemini_client = clients[
    "gemini_client"
]  # Optional. Initialize separately if you want Gemini visual fallback.

SUB_CLIENT = (
    None  # Did the client send data from multiple sites that that want to be analysed separately?
)
CBORD = None  # Set to True for CBORD data format, False for regular PDF extraction
assert CBORD is not None, "Please indicate whether data is from CBORD."

# dynamically set the client, whether it's baseline or pilot, and whether its procurement or serving
config = detect_client_structure(has_sub_client=SUB_CLIENT, step="prepare_pdf")
client = config["client"]
analysis_context = config["analysis_context"]
procurement_serving = config["procurement_serving"]
sub_client_name = config["sub_client_name"]
base_filepath = config["base_filepath"]
output_file = config["output_file"]

# Save metadata for subsequent notebooks
save_client_metadata(config, ENV_PATH, pdf_extracted=True)

# First step is always to have a look at the data manually, just give it a skim.
- Particularly, do they have the entry "add protein" ? that might been sorting

IF USING CBORD DATA: on one client we had an issue where the data is wide pivotted such as it has 1 col per day of the week. The fuzziness of the data made it hard to extract all those columns, so we just extract the "total" column at the end. However if you tell it in the prompt to infer the date from the text, it will sometimes screw up and actually pull all the individuals days alongside the weekly total. So later to process it, you'd need to xrop duplicates based on item name and portion, keeping the first one because the first one is always the total. 
SUGGESTION: when writing the extraction prompt for CBORD data, be more explicit in teling it where to find the date, so it doesn't do this.

In [ ]:
data_location = "raw_data"  # if this doesn't work maybe add a slash
pdf_files = [
    f for f in os.listdir(data_location) if f.endswith(".pdf")
]  # Get all pdfs in the data location

COLUMNS_TO_EXTRACT = ["Item Name", "Portion", "Total", "date"]
numeric_columns = ["Total"]  # columns in the data that should be numbers.
PRODUCT_NAME_COLUMN = "Item Name"

EXTRACTION_PROFILE = "auto"
PARSER_MODE = "auto"
AUTO_REPAIR = True
ENABLE_GEMINI_FALLBACK = True
AUDIT_THRESHOLD = 0.90
VISUAL_AUDIT_THRESHOLD = 0.75
TUNING_SAMPLE_CAP = 10
TUNING_RANDOM_SEED = 42

# Load the appropriate prompt based on CBORD setting
if CBORD:
    prompt_file = package_module_dir / "prompts" / "cbord_pdf_extraction_prompt.md"
else:
    prompt_file = package_module_dir / "prompts" / "standard_pdf_extraction_prompt.md"

with open(prompt_file) as f:
    prompt_template = f.read()

parse_extracted_pdf_prompt = prompt_template.format(columns_to_extract=COLUMNS_TO_EXTRACT)
print(parse_extracted_pdf_prompt)

## Preflight sample


In [ ]:
preflight_results = run_pdf_preflight_sample(
    pdf_files,
    whisper_client=whisper_client,
    OpenAI_client=OpenAI_client,
    data_location=data_location,
    product_name_column=PRODUCT_NAME_COLUMN,
    desired_columns=COLUMNS_TO_EXTRACT,
    numeric_columns=numeric_columns,
    parse_extracted_pdf_prompt=parse_extracted_pdf_prompt,
    CBORD=CBORD,
    extraction_profile=EXTRACTION_PROFILE,
    parser_mode=PARSER_MODE,
    tuning_sample_cap=TUNING_SAMPLE_CAP,
    tuning_random_seed=TUNING_RANDOM_SEED,
    audit_threshold=AUDIT_THRESHOLD,
    visual_audit_threshold=VISUAL_AUDIT_THRESHOLD,
    gemini_client=gemini_client,
    enable_gemini_fallback=ENABLE_GEMINI_FALLBACK,
)
preflight_results

## Run pipeline

In [ ]:
extraction_summary = run_pdf_extraction_pipeline_multiple_files(
    pdf_files,
    whisper_client=whisper_client,
    OpenAI_client=OpenAI_client,
    data_location=data_location,  # Where to find the file to extract
    product_name_column=PRODUCT_NAME_COLUMN,
    desired_columns=COLUMNS_TO_EXTRACT,
    numeric_columns=numeric_columns,
    parse_extracted_pdf_prompt=parse_extracted_pdf_prompt,
    debug=False,
    save_debug_artifacts=False,
    CBORD=CBORD,
    extraction_profile=EXTRACTION_PROFILE,
    parser_mode=PARSER_MODE,
    auto_repair=AUTO_REPAIR,
    enable_gemini_fallback=ENABLE_GEMINI_FALLBACK,
    gemini_client=gemini_client,
    audit_threshold=AUDIT_THRESHOLD,
    visual_audit_threshold=VISUAL_AUDIT_THRESHOLD,
    tuning_sample_cap=TUNING_SAMPLE_CAP,
    tuning_random_seed=TUNING_RANDOM_SEED,
    max_concurrent_files=6,
)

## Pull files together

In [ ]:
intended_columns = [*COLUMNS_TO_EXTRACT, "page", "original_file"]
full_data = combine_extracted_pdf_pages(data_location, intended_columns)

In [ ]:
full_data.head()

In [ ]:
check_duplicates(full_data)

# Create useful aggregations of the data for diagnostic plots

In [ ]:
pdf_validation_summary = summarize_pdf_validation_data(full_data)

file_counts = pdf_validation_summary["file_counts"]
page_counts = pdf_validation_summary["page_counts"]
page_file_counts = pdf_validation_summary["page_file_counts"]
plot_page_file_counts = pdf_validation_summary["plot_page_file_counts"]
file_summary = pdf_validation_summary["file_summary"]

# Check for missing data

In [ ]:
missing_rows = full_data[full_data.isna().any(axis=1)]

if missing_rows.empty:
    print("No missing data found in full_data.")
else:
    print(f"Found {len(missing_rows)} rows with missing data.")
    print("\nMissing values by column:")
    print(full_data.isna().sum()[full_data.isna().sum() > 0])
    print("\nRows with missing data:")
    print(missing_rows)
    raise AssertionError("Missing data found in full_data.")

In [ ]:
if CBORD:
    rows_with_missing_literal = check_missing_literal_values(full_data)
    if not rows_with_missing_literal.empty:
        print(rows_with_missing_literal.head(20).to_string(index=False))
else:
    print("Skipping exact 'missing' literal check because CBORD is False.")

# Diagnostic plots

In plot_rows_by_page_for_each_pdf, every PDF file gets its own line, and we plot the number of rows on each page. Ideally, we're looking for two things:
1. All lines overlap, indicating that each file has the same number of rows on each page. 
2. All lines are straight with no serious bumps or dips. This indicates that the number of rows for a given file on each page is about the same, which is what we would expect. 

We exclude the last page within each PDF, because the last page is often not full and so lower row counts there are usually expected rather than suspicious.

In [ ]:
plot_rows_by_page_for_each_pdf(
    page_file_counts=plot_page_file_counts,
    value_column="row_count",
    title="Row Count by Page Number for Each PDF",
    y_label="Rows on Page",
    subtitle="The last page for each PDF is excluded from this plot.",
)

In plot_rows_by_page_for_each_pdf, every PDF file gets its own line, and we plot the deviation of each page from the median number of rows per page for that file. This forces each line to centre around 0, and deviations are much clearer. A deviation here indicates that a given page has many more or fewer rows than other pages in the same file.

We exclude the last page within each PDF here as well, because final pages are often short for harmless reasons.

In [ ]:
plot_rows_by_page_for_each_pdf(
    page_file_counts=plot_page_file_counts,
    value_column="deviation_from_pdf_median",
    title="Deviation from Each PDF's Median Row Count by Page Number",
    y_label="Rows Relative to PDF Median",
    add_zero_line=True,
    subtitle=(
        "Each line is normalized to that PDF's own median row count.\n"
        "The last page for each PDF is excluded from this plot."
    ),
)

This prints the exact file x page combinations that deviate most from each PDF's own median row count. It is the table version of the plot above, so if you see a dip or spike in the chart, you can identify the specific file and page here.

In [ ]:
find_most_deviating_file_page_combos(
    page_file_counts=page_file_counts,
    n=10,
    exclude_last_page=True,
)

In [ ]:
plot_rowcount_against_pagenum(full_data)

# Do all files have similar numbers of rows?
We'd expect every file to have roughly similar numbers of rows. If we have extracted a much larger number of rows from one file compared to others, it might be because the extraction has malfunctioned and extracted lots of duplicates. Likewise, if one file is showing as having a tiny number of rows extracted compared to the other files, that might indicate that the extractor failed to extract most of the rows that are in the data.

In any case, you should go and look at the file to verify everything is fine. We plot each file's maximum page number against its extracted row count, because files with fewer pages should generally have fewer extracted rows. Files that sit well away from the overall diagonal trend deserve a manual check.

In [ ]:
plot_bar_by_pdf(
    file_summary=file_summary,
    value_column="total_rows",
    title="Total Row Count by PDF",
    y_label="Total Rows",
    color=GBD_colors[0],
)

In [ ]:
plot_bar_by_pdf(
    file_summary=file_summary,
    value_column="total_pages",
    title="Total Pages by PDF",
    y_label="Total Pages",
    color=GBD_colors[1],
)

check_high_duplicate_pages will see how many rows we have from each page. Most pages should have similar amounts of data across pages. The page that is commonly the last page will often have less data. 

check_high_duplicate_pages will look to see if any specific page that we extracted has a huge number of duplicates. This is a common failure mode of the PDF extractor.

In [ ]:
check_high_duplicate_pages(full_data, product_name_col=PRODUCT_NAME_COLUMN.lower())

# Do any pages have abnormal numbers of rows? 
qThis check is a little bit weird, but it makes sense when you think about it. We total up the number of rows in the first page of each file. Then we do the same for the second page, and the third, and the fourth, and so on. These totals should be roughly the same for all pages. Why would we expect page 4 to have more rows of data than page 7, for example? This is an indirect way of measuring whether there have been some failed extractions. 

Note that it is natural for pages close to the end of the document to have fewer rows than ones at the start. The first five pages of a document should probably all have the same number of rows. However, if documents are typically 30 pages long, but some of them are 29 and some are 28 pages long, we would expect that the sum total of 

In [ ]:
plot_total_rows_by_page_across_pdfs(page_counts.sort_index())

In [ ]:
check_extraction_by_page(full_data)

# Spot check if you want to be extra careful
Lastly, sample_files_and_rows will pick five random files and it will sample five consecutive rows from each file and print them out so you can then go and look at them manually to just spot check a few of the extractions.

In [ ]:
sample_files_and_rows(full_data, num_files=5, rows_per_file=5)

# Under construction
Sometimes the PDF extractor does not extract the names of products properly. For example it might see the product "Beyond Burger" twice in the data, but one time extract as "Beynd Burger" and the other time "Beyond Burgr". This isn't the end of the world but means we can't actually do "most/least popular product" analysis, because they aren't recorded in the data as the same name.

In [ ]:
PRODUCT_NAME_COLUMN = PRODUCT_NAME_COLUMN.lower()

full_data[PRODUCT_NAME_COLUMN] = full_data[PRODUCT_NAME_COLUMN].str.replace(
    "*", "", regex=False
)  # common data cleaning issue pdf data requires
full_data[PRODUCT_NAME_COLUMN] = full_data[PRODUCT_NAME_COLUMN].str.strip()

possible_misspellings = find_possible_misspellings(
    full_data, threshold=95, product_name_col=PRODUCT_NAME_COLUMN
)
print(find_possible_misspellings(full_data, threshold=80, product_name_col=PRODUCT_NAME_COLUMN))
print(
    identify_unique_product_names(full_data, product_name_col=PRODUCT_NAME_COLUMN)
)  # None of these indicate failed extraction causing a misspelling

In [ ]:
full_data.to_csv(output_file, index=False)

In [ ]:
# Run step 1 categorization on the CSV produced in this notebook
script_path = package_module_dir.parents[0] / "Customer template" / "1. Categorize Runscript.py"
if not script_path.exists():
    raise FileNotFoundError(f"Categorize runscript not found at: {script_path}")

data_type = "serving" if "serving" in procurement_serving.lower() else "procurement"
cmd = [
    sys.executable,
    str(script_path),
    "--input",
    str(output_file),
    "--analysis-context",
    analysis_context,
    "--data-type",
    data_type,
]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)